# Exploracao do potencial de exportacoes por UF

Este notebook explora os dados monetarios do modelo EPI para comparar SC com outras UFs nos produtos SH6 020714 e 440711.

Todas as metricas sao calculadas no nivel do par produto-pais por UF exportadora.

Foco da analise:
- potencial de exportacoes
- exportacoes atuais estimadas
- potencial nao realizado

In [1]:
from pathlib import Path
import polars as pl
import pandas as pd
import plotly.express as px

pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [2]:
def resolve_path(relative_path: str) -> Path:
    candidates = [
        Path(relative_path),
        Path('..') / relative_path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Arquivo nao encontrado: {relative_path}')

path_detail = resolve_path('data/processed/epi_monetary_ufs.parquet')
path_country = resolve_path('data/processed/epi_monetary_ufs_country.parquet')

path_detail, path_country

(WindowsPath('../data/processed/epi_monetary_ufs.parquet'),
 WindowsPath('../data/processed/epi_monetary_ufs_country.parquet'))

In [3]:
selected_sh6 = ['020714', '440711']

df_detail_raw = pl.read_parquet(path_detail)
if 'product_description' not in df_detail_raw.columns and 'product_description_br' in df_detail_raw.columns:
    df_detail_raw = df_detail_raw.rename({'product_description_br': 'product_description'})

df_pair = (
    df_detail_raw
    .with_columns(pl.col('sh6').cast(pl.Utf8).str.zfill(6))
    .filter(pl.col('sh6').is_in(selected_sh6))
    .with_columns((pl.col('potential_value') * pl.col('potential_utilization_ratio')).alias('current_exports_value'))
    .with_columns(pl.when(pl.col('current_exports_value') < 0).then(0.0).otherwise(pl.col('current_exports_value')).alias('current_exports_value'))
    .rename({'sg_uf': 'exportador_uf', 'importer': 'pais_parceiro'})
    .select([
        'exportador_uf', 'pais_parceiro', 'sh6', 'product_description',
        'potential_value', 'current_exports_value', 'unrealized_potential_value'
    ])
)

df_pair.sort(['sh6', 'unrealized_potential_value'], descending=[False, True]).head(30)

exportador_uf,pais_parceiro,sh6,product_description,potential_value,current_exports_value,unrealized_potential_value
str,str,str,str,f64,f64,f64
"""PR""","""CHN""","""020714""","""Pedaços e miudezas comestíveis…",7.5124e8,5.5508e8,1.9616e8
"""PR""","""CHL""","""020714""","""Pedaços e miudezas comestíveis…",1.5420e8,6.3666e7,9.0534e7
"""SC""","""CHL""","""020714""","""Pedaços e miudezas comestíveis…",1.0424e8,3.5916e7,6.8325e7
"""RS""","""CHN""","""020714""","""Pedaços e miudezas comestíveis…",1.9596e8,1.4551e8,5.0448e7
"""SC""","""CHN""","""020714""","""Pedaços e miudezas comestíveis…",3.7128e8,3.2110e8,5.0182e7
…,…,…,…,…,…,…
"""MS""","""CHN""","""020714""","""Pedaços e miudezas comestíveis…",8.1811e7,6.1702e7,2.0109e7
"""SC""","""NLD""","""020714""","""Pedaços e miudezas comestíveis…",3.4499e7,1.4608e7,1.9891e7
"""RS""","""AGO""","""020714""","""Pedaços e miudezas comestíveis…",2.5144e7,5.2945e6,1.9849e7


In [4]:
df_exportador_pais = (
    df_pair
    .group_by(['exportador_uf', 'pais_parceiro', 'sh6', 'product_description'])
    .agg([
        pl.sum('potential_value').alias('potential_value'),
        pl.sum('current_exports_value').alias('current_exports_value'),
        pl.sum('unrealized_potential_value').alias('unrealized_potential_value'),
    ])
    .sort(['sh6', 'unrealized_potential_value'], descending=[False, True])
)

df_exportador_pais_pd = df_exportador_pais.to_pandas()
df_exportador_pais_pd.head(40)

df_exportador_pais_pd

,exportador_uf,pais_parceiro,sh6,product_description,potential_value,current_exports_value,unrealized_potential_value
0,PR,CHN,020714,Pedaços e miudezas comestíveis de galos e gali...,"751,237,909.34","555,077,437.13","196,160,472.21"
1,PR,CHL,020714,Pedaços e miudezas comestíveis de galos e gali...,"154,200,615.44","63,666,362.06","90,534,253.38"
2,SC,CHL,020714,Pedaços e miudezas comestíveis de galos e gali...,"104,241,181.28","35,916,382.58","68,324,798.71"
3,RS,CHN,020714,Pedaços e miudezas comestíveis de galos e gali...,"195,957,444.86","145,509,575.06","50,447,869.80"
4,SC,CHN,020714,Pedaços e miudezas comestíveis de galos e gali...,"371,283,827.66","321,101,696.80","50,182,130.85"
...,...,...,...,...,...,...,...
11497,PA,ITA,440711,"Madeira serrada ou fendida longitudinalmente, ...",0.00,0.00,0.00
11498,RN,MWI,440711,"Madeira serrada ou fendida longitudinalmente, ...",0.00,0.00,0.00
11499,BA,ZMB,440711,"Madeira serrada ou fendida longitudinalmente, ...",0.00,0.00,0.00
11500,TO,SAU,440711,"Madeira serrada ou fendida longitudinalmente, ...",0.00,0.00,0.00


In [5]:
top_parceiros = (
    df_exportador_pais
    .group_by(['sh6', 'pais_parceiro'])
    .agg(pl.sum('potential_value').alias('potential_value'))
    .sort(['sh6', 'potential_value'], descending=[False, True])
    .group_by('sh6')
    .head(8)
    .select(['sh6', 'pais_parceiro'])
)

df_sc_vs_outros = (
    df_exportador_pais
    .join(top_parceiros, on=['sh6', 'pais_parceiro'], how='inner')
    .with_columns(
        pl.when(pl.col('exportador_uf') == 'SC').then(pl.lit('SC')).otherwise(pl.lit('Outras UFs')).alias('grupo_uf')
    )
    .group_by(['sh6', 'product_description', 'pais_parceiro', 'grupo_uf'])
    .agg([
        pl.sum('potential_value').alias('potential_value'),
        pl.sum('current_exports_value').alias('current_exports_value'),
        pl.sum('unrealized_potential_value').alias('unrealized_potential_value'),
    ])
)

df_long = (
    df_sc_vs_outros
    .melt(
        id_vars=['sh6', 'product_description', 'pais_parceiro', 'grupo_uf'],
        value_vars=['potential_value', 'current_exports_value', 'unrealized_potential_value'],
        variable_name='indicador',
        value_name='valor'
    )
)

fig = px.bar(
    df_long.to_pandas(),
    x='grupo_uf',
    y='valor',
    color='grupo_uf',
    facet_row='sh6',
    facet_col='indicador',
    animation_frame='pais_parceiro',
    barmode='group',
    title='SC vs outras UFs por pais parceiro: potencial, exportacoes atuais e potencial nao realizado',
    labels={'valor': 'Valor (USD)', 'grupo_uf': 'Grupo', 'pais_parceiro': 'Pais parceiro'}
)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace('sh6=', 'SH6 ').replace('indicador=', '')))
fig.update_layout(height=900, width=1200, showlegend=False)
fig.show()

C:\Users\ailton-junior\AppData\Local\Temp\ipykernel_25848\1385216530.py:27: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  .melt(


In [6]:
top_pares = (
    df_exportador_pais
    .sort('unrealized_potential_value', descending=True)
    .group_by('sh6')
    .head(15)
    .sort(['sh6', 'unrealized_potential_value'], descending=[False, True])
)

fig2 = px.bar(
    top_pares.to_pandas(),
    x='exportador_uf',
    y='unrealized_potential_value',
    color='pais_parceiro',
    facet_col='sh6',
    title='Top pares UF-pais por potencial nao realizado',
    labels={
        'exportador_uf': 'UF exportadora',
        'pais_parceiro': 'Pais parceiro',
        'unrealized_potential_value': 'Potencial nao realizado (USD)'
    }
)
fig2.for_each_annotation(lambda a: a.update(text=a.text.replace('sh6=', 'SH6 ')))
fig2.update_layout(height=550, width=1250)
fig2.show()

top_pares.to_pandas()

,sh6,exportador_uf,pais_parceiro,product_description,potential_value,current_exports_value,unrealized_potential_value
0,020714,PR,CHN,Pedaços e miudezas comestíveis de galos e gali...,"751,237,909.34","555,077,437.13","196,160,472.21"
1,020714,PR,CHL,Pedaços e miudezas comestíveis de galos e gali...,"154,200,615.44","63,666,362.06","90,534,253.38"
2,020714,SC,CHL,Pedaços e miudezas comestíveis de galos e gali...,"104,241,181.28","35,916,382.58","68,324,798.71"
3,020714,RS,CHN,Pedaços e miudezas comestíveis de galos e gali...,"195,957,444.86","145,509,575.06","50,447,869.80"
4,020714,SC,CHN,Pedaços e miudezas comestíveis de galos e gali...,"371,283,827.66","321,101,696.80","50,182,130.85"
5,020714,PR,NLD,Pedaços e miudezas comestíveis de galos e gali...,"64,567,643.43","25,388,619.56","39,179,023.87"
6,020714,PR,USA,Pedaços e miudezas comestíveis de galos e gali...,"38,339,714.80","257,193.11","38,082,521.69"
7,020714,SC,HKG,Pedaços e miudezas comestíveis de galos e gali...,"74,325,188.65","36,983,768.22","37,341,420.43"
8,020714,PR,VNM,Pedaços e miudezas comestíveis de galos e gali...,"50,781,570.88","13,943,606.03","36,837,964.86"
9,020714,PR,AGO,Pedaços e miudezas comestíveis de galos e gali...,"52,926,250.31","20,066,580.82","32,859,669.49"


## Leitura rapida dos resultados

A unidade analitica do notebook e sempre o par produto-pais por UF exportadora.

- potential_value: potencial monetario do par (UF, SH6, pais parceiro)
- current_exports_value: exportacao atual estimada do mesmo par
- unrealized_potential_value: potencial nao realizado do mesmo par

Sugestao: priorizar os pares UF-pais com maior potencial nao realizado para cada SH6, com destaque para SC.

## Exportacao da tabela Brasil x Share UF

A tabela abaixo e exportada com as colunas:
- exportacoes_brasil
- sg_uf
- share_uf
- valor_exportado_uf (exportacoes_brasil x share_uf)

In [7]:
from pathlib import Path
import polars as pl

def resolve_path(relative_path: str) -> Path:
    candidates = [
        Path(relative_path),
        Path('..') / relative_path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Arquivo nao encontrado: {relative_path}')

path_bilateral = resolve_path('data/interim/bilateral_exports_sh6.parquet')
path_products = resolve_path('data/interim/comex_exps_weighted.parquet')
path_countries = resolve_path('references/countries_br.csv')
out_parquet = resolve_path('data/processed') / 'exportacoes_brasil_share_uf.parquet'
out_csv = resolve_path('data/processed') / 'exportacoes_brasil_share_uf.csv'

df_products = (
    pl.read_parquet(path_products)
    .filter((pl.col('exporter') == 'BRA') & (pl.col('year') == 2024))
    .select(['sh6', 'product_description'])
    .unique(subset=['sh6'])
)

df_countries = (
    pl.read_csv(path_countries, encoding='latin1', separator=';')
    .select([
        pl.col('CO_PAIS_ISOA3').alias('importer'),
        pl.col('NO_PAIS').alias('pais_destino'),
    ])
)

df_exportacoes_uf = (
    pl.read_parquet(path_bilateral)
    .filter(pl.col('exporter') == 'BRA')
    .with_columns(pl.lit(2024).alias('year'))
    .with_columns(pl.col('sh6').cast(pl.Int64))
    .join(df_products, on='sh6', how='left')
    .join(df_countries, on='importer', how='left')
    .with_columns(
        pl.sum('bilateral_exports_uf_sh6').over(['year', 'importer', 'sh6']).alias('exportacoes_brasil')
    )
    .with_columns(
        pl.when(pl.col('exportacoes_brasil') > 0)
        .then(pl.col('bilateral_exports_uf_sh6') / pl.col('exportacoes_brasil'))
        .otherwise(0.0)
        .alias('share_uf')
    )
    .with_columns(
        (pl.col('exportacoes_brasil') * pl.col('share_uf')).alias('valor_exportado_uf')
    )
    .select([
        'year',
        'importer',
        'pais_destino',
        'sh6',
        'product_description',
        'exportacoes_brasil',
        'sg_uf',
        'share_uf',
        'valor_exportado_uf',
    ])
    .sort(['year', 'importer', 'sh6', 'sg_uf'])
)

df_exportacoes_uf.write_parquet(out_parquet)
df_exportacoes_uf.write_csv(out_csv)

print(f'Arquivo parquet exportado: {out_parquet}')
print(f'Arquivo csv exportado: {out_csv}')
print('Linhas:', df_exportacoes_uf.height)

df_exportacoes_uf.head(20)

Arquivo parquet exportado: ..\data\processed\exportacoes_brasil_share_uf.parquet
Arquivo csv exportado: ..\data\processed\exportacoes_brasil_share_uf.csv
Linhas: 4363841


year,importer,pais_destino,sh6,product_description,exportacoes_brasil,sg_uf,share_uf,valor_exportado_uf
i32,str,str,i64,str,f64,str,f64,f64
2024,"""ABW""","""Aruba""",20130,"""Meat: of bovine animals, bonel…",2.0228e6,"""AC""",0.000377,762.694787
2024,"""ABW""","""Aruba""",20130,"""Meat: of bovine animals, bonel…",2.0228e6,"""AL""",0.000022,45.215503
2024,"""ABW""","""Aruba""",20130,"""Meat: of bovine animals, bonel…",2.0228e6,"""AM""",0.000019,38.971543
2024,"""ABW""","""Aruba""",20130,"""Meat: of bovine animals, bonel…",2.0228e6,"""AP""",0.00001,20.935401
2024,"""ABW""","""Aruba""",20130,"""Meat: of bovine animals, bonel…",2.0228e6,"""BA""",0.011264,22785.145222
…,…,…,…,…,…,…,…,…
2024,"""ABW""","""Aruba""",20130,"""Meat: of bovine animals, bonel…",2.0228e6,"""PE""",0.000009,17.663191
2024,"""ABW""","""Aruba""",20130,"""Meat: of bovine animals, bonel…",2.0228e6,"""PI""",0.0,0.0
2024,"""ABW""","""Aruba""",20130,"""Meat: of bovine animals, bonel…",2.0228e6,"""PR""",0.019044,38523.330262


## Comparacao SC-only vs SC no fluxo UFs

Esta secao exporta uma planilha comparando o potencial monetario por SH6 entre:
- fluxo SC-only (`epi_monetary_sc_sh6.json`)
- fluxo UFs filtrado para SC (`epi_monetary_ufs_sh6.json`)

Saidas geradas em `data/processed`:
- `comparacao_potencial_sc_vs_ufs_sc.xlsx`
- `comparacao_potencial_sc_vs_ufs_sc.csv`

In [8]:
from pathlib import Path
import polars as pl

def resolve_existing_path(candidates: list[str]) -> Path:
    bases = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for rel in candidates:
        rel_path = Path(rel)
        if rel_path.is_absolute() and rel_path.exists():
            return rel_path
        for base in bases:
            candidate = (base / rel_path).resolve()
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'Nao foi possivel localizar nenhum dos caminhos: {candidates}')

path_ufs = resolve_existing_path([
    'data/processed/epi_monetary_ufs_sh6.parquet',
    '../data/processed/epi_monetary_ufs_sh6.parquet',
])

path_sc = resolve_existing_path([
    '../../sc_only_compare_20260416/data/processed/epi_monetary_sc_sh6.json',
    '../sc_only_compare_20260416/data/processed/epi_monetary_sc_sh6.json',
    'sc_only_compare_20260416/data/processed/epi_monetary_sc_sh6.json',
])

df_ufs_sc = (
    pl.read_parquet(path_ufs)
    .filter(pl.col('sg_uf') == 'SC')
    .with_columns(pl.col('sh6').cast(pl.Utf8).str.zfill(6))
    .select(['sh6', 'potential_value', 'unrealized_potential_value'])
    .rename({
        'potential_value': 'potential_ufs_sc',
        'unrealized_potential_value': 'unrealized_ufs_sc',
    })
)

df_sc_only = (
    pl.read_json(path_sc)
    .with_columns(pl.col('sh6').cast(pl.Utf8).str.zfill(6))
    .select(['sh6', 'potential_value', 'unrealized_potential_value'])
    .rename({
        'potential_value': 'potential_sc_only',
        'unrealized_potential_value': 'unrealized_sc_only',
    })
)

df_comp = (
    df_sc_only
    .join(df_ufs_sc, on='sh6', how='inner')
    .with_columns([
        (pl.col('potential_ufs_sc') - pl.col('potential_sc_only')).alias('diff_potential'),
        (pl.col('unrealized_ufs_sc') - pl.col('unrealized_sc_only')).alias('diff_unrealized'),
        pl.when(pl.col('potential_sc_only') != 0)
          .then(pl.col('potential_ufs_sc') / pl.col('potential_sc_only') - 1)
          .otherwise(None)
          .alias('pct_diff_potential'),
        pl.when(pl.col('unrealized_sc_only') != 0)
          .then(pl.col('unrealized_ufs_sc') / pl.col('unrealized_sc_only') - 1)
          .otherwise(None)
          .alias('pct_diff_unrealized'),
    ])
    .sort('sh6')
)

out_dir = path_ufs.parent
out_xlsx = out_dir / 'comparacao_potencial_sc_vs_ufs_sc.xlsx'
out_csv = out_dir / 'comparacao_potencial_sc_vs_ufs_sc.csv'

df_comp.write_csv(out_csv)
df_comp.to_pandas().to_excel(out_xlsx, index=False)

resumo = pl.DataFrame({
    'metrica': ['soma_sc_only_potential', 'soma_ufs_sc_potential', 'pct_total_potential'],
    'valor': [
        df_comp['potential_sc_only'].sum(),
        df_comp['potential_ufs_sc'].sum(),
        (df_comp['potential_ufs_sc'].sum() / df_comp['potential_sc_only'].sum()) - 1,
    ],
})

print(f'Arquivo xlsx exportado: {out_xlsx}')
print(f'Arquivo csv exportado: {out_csv}')
resumo
df_comp.head(20)

FileNotFoundError: Nao foi possivel localizar nenhum dos caminhos: ['../../sc_only_compare_20260416/data/processed/epi_monetary_sc_sh6.json', '../sc_only_compare_20260416/data/processed/epi_monetary_sc_sh6.json', 'sc_only_compare_20260416/data/processed/epi_monetary_sc_sh6.json']

## Exportacao do potencial por SH6 e pais importador

Esta secao exporta uma planilha com o potencial agregado por SH6 e pais importador.

Saidas geradas em `data/processed`:
- `potencial_por_sh6_importador.xlsx`
- `potencial_por_sh6_importador.csv`

In [9]:
from pathlib import Path
import polars as pl

def resolve_existing_path(candidates: list[str]) -> Path:
    bases = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for rel in candidates:
        rel_path = Path(rel)
        if rel_path.is_absolute() and rel_path.exists():
            return rel_path
        for base in bases:
            candidate = (base / rel_path).resolve()
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'Nao foi possivel localizar nenhum dos caminhos: {candidates}')

path_detail_ufs = resolve_existing_path([
    'data/processed/epi_monetary_ufs.parquet',
    '../data/processed/epi_monetary_ufs.parquet',
])

df_potencial_sh6_importador = (
    pl.read_parquet(path_detail_ufs)
    .with_columns([
        pl.col('sh6').cast(pl.Utf8).str.zfill(6).alias('sh6'),
        pl.col('importer').cast(pl.Utf8).alias('importer'),
    ])
    .group_by(['sh6', 'importer'])
    .agg([
        pl.first('product_description_br').alias('product_description'),
        pl.sum('potential_value').alias('potential_value'),
    ])
    .sort(['sh6', 'potential_value'], descending=[False, True])
)

out_dir = path_detail_ufs.parent
out_xlsx = out_dir / 'potencial_por_sh6_importador.xlsx'
out_csv = out_dir / 'potencial_por_sh6_importador.csv'

df_potencial_sh6_importador.write_csv(out_csv)
df_potencial_sh6_importador.to_pandas().to_excel(out_xlsx, index=False)

print(f'Arquivo xlsx exportado: {out_xlsx}')
print(f'Arquivo csv exportado: {out_csv}')
print(f'Linhas exportadas: {df_potencial_sh6_importador.height}')

df_potencial_sh6_importador.head(20)

Arquivo xlsx exportado: C:\Users\ailton-junior\2.Teste\export_potential\data\processed\potencial_por_sh6_importador.xlsx
Arquivo csv exportado: C:\Users\ailton-junior\2.Teste\export_potential\data\processed\potencial_por_sh6_importador.csv
Linhas exportadas: 833300


sh6,importer,product_description,potential_value
str,str,str,f64
"""010121""","""GBR""","""Cavalos reprodutores de raça p…",625660.302376
"""010121""","""USA""","""Cavalos reprodutores de raça p…",529470.511797
"""010121""","""IRL""","""Cavalos reprodutores de raça p…",251067.153719
"""010121""","""JPN""","""Cavalos reprodutores de raça p…",206457.286383
"""010121""","""ARG""","""Cavalos reprodutores de raça p…",144559.314868
…,…,…,…
"""010121""","""CHL""","""Cavalos reprodutores de raça p…",30920.657633
"""010121""","""PRY""","""Cavalos reprodutores de raça p…",26766.513946
"""010121""","""DNK""","""Cavalos reprodutores de raça p…",26284.571817


## Comparacao do potencial no par produto-pais (SC-only vs UFs-SC)

Esta secao compara o potencial por par SH6-importador entre:
- fluxo SC-only
- fluxo UFs filtrado para SC

A comparacao considera o potencial agregado por par produto-pais.

In [10]:
from pathlib import Path
import polars as pl

def resolve_existing_path(candidates: list[str]) -> Path:
    bases = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for rel in candidates:
        rel_path = Path(rel)
        if rel_path.is_absolute() and rel_path.exists():
            return rel_path
        for base in bases:
            candidate = (base / rel_path).resolve()
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'Nao foi possivel localizar nenhum dos caminhos: {candidates}')

path_ufs_detail = resolve_existing_path([
    'data/processed/epi_monetary_ufs.parquet',
    '../data/processed/epi_monetary_ufs.parquet',
])

path_sc_detail = resolve_existing_path([
    '../../sc_only_compare_20260416/data/processed/epi_monetary_sc.json',
    '../sc_only_compare_20260416/data/processed/epi_monetary_sc.json',
    'sc_only_compare_20260416/data/processed/epi_monetary_sc.json',
])

df_ufs_pair = (
    pl.read_parquet(path_ufs_detail)
    .filter(pl.col('sg_uf') == 'SC')
    .with_columns([
        pl.col('sh6').cast(pl.Utf8).str.zfill(6).alias('sh6'),
        pl.col('importer').cast(pl.Utf8).alias('importer'),
    ])
    .group_by(['sh6', 'importer'])
    .agg([
        pl.first('product_description_br').alias('product_description_ufs'),
        pl.sum('potential_value').alias('potential_ufs_sc_pair'),
    ])
)

df_sc_pair = (
    pl.read_json(path_sc_detail)
    .with_columns([
        pl.col('sh6').cast(pl.Utf8).str.zfill(6).alias('sh6'),
        pl.col('importer').cast(pl.Utf8).alias('importer'),
    ])
    .group_by(['sh6', 'importer'])
    .agg([
        pl.first('product_description').alias('product_description_sc'),
        pl.sum('potential_value').alias('potential_sc_only_pair'),
    ])
)

df_pair_cmp = (
    df_sc_pair
    .join(df_ufs_pair, on=['sh6', 'importer'], how='inner')
    .with_columns([
        pl.coalesce([pl.col('product_description_sc'), pl.col('product_description_ufs')]).alias('product_description'),
        (pl.col('potential_ufs_sc_pair') - pl.col('potential_sc_only_pair')).alias('diff_potential'),
        pl.when(pl.col('potential_sc_only_pair') != 0)
          .then(pl.col('potential_ufs_sc_pair') / pl.col('potential_sc_only_pair') - 1)
          .otherwise(None)
          .alias('pct_diff_potential'),
    ])
    .select([
        'sh6', 'importer', 'product_description',
        'potential_sc_only_pair', 'potential_ufs_sc_pair',
        'diff_potential', 'pct_diff_potential',
    ])
    .sort('diff_potential', descending=True)
)

df_only_sc = df_sc_pair.join(df_ufs_pair.select(['sh6', 'importer']), on=['sh6', 'importer'], how='anti')
df_only_ufs = df_ufs_pair.join(df_sc_pair.select(['sh6', 'importer']), on=['sh6', 'importer'], how='anti')

sum_sc = float(df_pair_cmp['potential_sc_only_pair'].sum())
sum_ufs = float(df_pair_cmp['potential_ufs_sc_pair'].sum())
pct_total = (sum_ufs / sum_sc - 1.0) if sum_sc != 0 else None

resumo_pair = pl.DataFrame({
    'metrica': [
        'pares_sc_only',
        'pares_ufs_sc',
        'pares_comuns',
        'pares_apenas_sc_only',
        'pares_apenas_ufs_sc',
        'soma_potential_sc_only',
        'soma_potential_ufs_sc',
        'pct_total_potential',
    ],
    'valor': [
        float(df_sc_pair.height),
        float(df_ufs_pair.height),
        float(df_pair_cmp.height),
        float(df_only_sc.height),
        float(df_only_ufs.height),
        sum_sc,
        sum_ufs,
        pct_total,
    ],
})

out_dir = path_ufs_detail.parent
out_pair_csv = out_dir / 'comparacao_potencial_par_produto_pais_sc_vs_ufs_sc.csv'
out_pair_xlsx = out_dir / 'comparacao_potencial_par_produto_pais_sc_vs_ufs_sc.xlsx'

df_pair_cmp.write_csv(out_pair_csv)
df_pair_cmp.to_pandas().to_excel(out_pair_xlsx, index=False)

print(f'Arquivo csv exportado: {out_pair_csv}')
print(f'Arquivo xlsx exportado: {out_pair_xlsx}')
resumo_pair
df_pair_cmp.head(20)

FileNotFoundError: Nao foi possivel localizar nenhum dos caminhos: ['../../sc_only_compare_20260416/data/processed/epi_monetary_sc.json', '../sc_only_compare_20260416/data/processed/epi_monetary_sc.json', 'sc_only_compare_20260416/data/processed/epi_monetary_sc.json']

## Validacao de fechamento SC

Esta secao valida o fechamento entre:
- fluxo SC-only
- fluxo UFs filtrado para SC

A validacao e feita em dois niveis:
- total de potencial
- par SH6-importador

In [11]:
from pathlib import Path
from collections import defaultdict
import json
import polars as pl


def resolve_existing_path(candidates: list[str]) -> Path:
    bases = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for rel in candidates:
        rel_path = Path(rel)
        if rel_path.is_absolute() and rel_path.exists():
            return rel_path
        for base in bases:
            candidate = (base / rel_path).resolve()
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'Nao foi possivel localizar nenhum dos caminhos: {candidates}')


path_ufs_detail = resolve_existing_path([
    'data/processed/epi_monetary_ufs.parquet',
    '../data/processed/epi_monetary_ufs.parquet',
])

path_sc_detail = resolve_existing_path([
    '../../sc_only_compare_20260416/data/processed/epi_monetary_sc.json',
    '../sc_only_compare_20260416/data/processed/epi_monetary_sc.json',
    'sc_only_compare_20260416/data/processed/epi_monetary_sc.json',
])

# Tolerancias numericas para operacoes de ponto flutuante
tol_pair = 1e-6
tol_total = 1e-5

# Leitura Parquet para UFs (mais eficiente que streaming JSON)
pair_ufs_map = defaultdict(float)
sum_ufs_sc = 0.0

df_ufs = pl.read_parquet(path_ufs_detail).filter(pl.col('sg_uf') == 'SC')
for row in df_ufs.iter_rows(named=True):
    sh6 = str(row['sh6']).zfill(6)
    importer = str(row['importer'])
    potential = float(row['potential_value'])
    pair_ufs_map[(sh6, importer)] += potential
    sum_ufs_sc += potential

# Leitura JSON para SC (mantém compatibilidade com arquivo original)
def iter_json_array(path: Path, chunk_size: int = 8 * 1024 * 1024):
    decoder = json.JSONDecoder()
    with path.open('r', encoding='utf-8') as f:
        while True:
            ch = f.read(1)
            if ch == '':
                return
            if ch.isspace():
                continue
            if ch != '[':
                raise ValueError(f'JSON invalido em {path}: esperado [ no inicio')
            break

        buffer = ''
        eof = False

        while True:
            if not eof:
                chunk = f.read(chunk_size)
                if chunk == '':
                    eof = True
                else:
                    buffer += chunk

            pos = 0
            parsed_any = False

            while True:
                while pos < len(buffer) and buffer[pos] in ' \t\n\r,':
                    pos += 1

                if pos >= len(buffer):
                    break

                if buffer[pos] == ']':
                    return

                try:
                    obj, next_pos = decoder.raw_decode(buffer, pos)
                except json.JSONDecodeError:
                    break

                yield obj
                parsed_any = True
                pos = next_pos

            buffer = buffer[pos:]

            if eof:
                tail = buffer.strip()
                if tail in ('', ']'):
                    return
                if not parsed_any:
                    raise ValueError(f'Fim inesperado ao parsear {path}')


pair_sc_map = defaultdict(float)
sum_sc_only = 0.0

for row in iter_json_array(path_sc_detail):
    sh6 = str(row.get('sh6')).zfill(6)
    importer = str(row.get('importer'))
    potential = float(row.get('potential_value') or 0.0)
    pair_sc_map[(sh6, importer)] += potential
    sum_sc_only += potential

pair_ufs = pl.DataFrame({
    'sh6': [k[0] for k in pair_ufs_map.keys()],
    'importer': [k[1] for k in pair_ufs_map.keys()],
    'potential_ufs_sc': [v for v in pair_ufs_map.values()],
})

pair_sc = pl.DataFrame({
    'sh6': [k[0] for k in pair_sc_map.keys()],
    'importer': [k[1] for k in pair_sc_map.keys()],
    'potential_sc_only': [v for v in pair_sc_map.values()],
})

pair_cmp = (
    pair_sc
    .join(pair_ufs, on=['sh6', 'importer'], how='full')
    .with_columns([
        pl.col('potential_sc_only').fill_null(0.0),
        pl.col('potential_ufs_sc').fill_null(0.0),
    ])
    .with_columns([
        (pl.col('potential_ufs_sc') - pl.col('potential_sc_only')).alias('diff_potential'),
        (pl.col('potential_ufs_sc') - pl.col('potential_sc_only')).abs().alias('diff_abs'),
    ])
)

total_diff = sum_ufs_sc - sum_sc_only
n_pairs = pair_cmp.height
n_div_tol = pair_cmp.filter(pl.col('diff_abs') > tol_pair).height
max_diff_abs = float(pair_cmp['diff_abs'].max()) if n_pairs > 0 else 0.0

status = 'OK' if (abs(total_diff) <= tol_total and n_div_tol == 0) else 'ATENCAO'

resumo_fechamento_sc = pl.DataFrame({
    'metrica': [
        'status',
        'tol_pair',
        'tol_total',
        'soma_sc_only',
        'soma_ufs_sc',
        'diff_total',
        'pares_comparados',
        'pares_diff_gt_tol_pair',
        'max_diff_abs',
    ],
    'valor': [
        status,
        f'{tol_pair:.2e}',
        f'{tol_total:.2e}',
        f'{sum_sc_only:.12f}',
        f'{sum_ufs_sc:.12f}',
        f'{total_diff:.12f}',
        str(n_pairs),
        str(n_div_tol),
        f'{max_diff_abs:.12f}',
    ],
})

print(f'Validacao fechamento SC: {status}')
print(f'soma_sc_only={sum_sc_only}')
print(f'soma_ufs_sc={sum_ufs_sc}')
print(f'diff_total={total_diff}')
print(f'pares_diff_gt_tol_pair={n_div_tol} de {n_pairs}')

resumo_fechamento_sc
pair_cmp.sort('diff_abs', descending=True).head(20)

FileNotFoundError: Nao foi possivel localizar nenhum dos caminhos: ['../../sc_only_compare_20260416/data/processed/epi_monetary_sc.json', '../sc_only_compare_20260416/data/processed/epi_monetary_sc.json', 'sc_only_compare_20260416/data/processed/epi_monetary_sc.json']

## Exportacao SC para validacao externa

Exporta os valores calculados para SC no fluxo com todas as UFs, em dois niveis:
- total por SH6
- detalhe por SH6 e parceiro importador

Metricas exportadas: potencial total, exportacoes atuais e potencial nao realizado.

In [ ]:
from pathlib import Path
import pandas as pd
import polars as pl


def resolve_existing_path(candidates: list[str]) -> Path:
    bases = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for rel in candidates:
        rel_path = Path(rel)
        if rel_path.is_absolute() and rel_path.exists():
            return rel_path
        for base in bases:
            candidate = (base / rel_path).resolve()
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'Nao foi possivel localizar nenhum dos caminhos: {candidates}')


path_detail_ufs = resolve_existing_path([
    'data/processed/epi_monetary_ufs.parquet',
    '../data/processed/epi_monetary_ufs.parquet',
])

out_dir = path_detail_ufs.parent
out_xlsx = out_dir / 'validacao_externa_sc_potencial_sh6_parceiro.xlsx'
out_csv_detail = out_dir / 'validacao_externa_sc_potencial_sh6_parceiro.csv'
out_csv_sh6 = out_dir / 'validacao_externa_sc_potencial_sh6_total.csv'

df_sc = (
    pl.read_parquet(path_detail_ufs)
    .filter(pl.col('sg_uf') == 'SC')
    .with_columns([
        pl.col('sh6').cast(pl.Utf8).str.zfill(6).alias('sh6'),
        pl.col('importer').cast(pl.Utf8).alias('importer'),
        pl.col('product_description_br').cast(pl.Utf8).alias('product_description'),
        pl.col('sc_comp').cast(pl.Utf8).alias('sc_comp'),
        pl.col('bilateral_exports_uf_sh6').fill_null(0.0).alias('exportacoes_sc'),
    ])
)

df_sc_detail = (
    df_sc
    .group_by(['sh6', 'product_description', 'sc_comp', 'importer'])
    .agg([
        pl.sum('potential_value').alias('potencial_total_sc'),
        pl.sum('exportacoes_sc').alias('exportacoes_sc'),
        pl.sum('unrealized_potential_value').alias('potencial_nao_realizado_sc'),
    ])
    .sort(['sh6', 'potencial_total_sc'], descending=[False, True])
)

df_sc_sh6 = (
    df_sc_detail
    .group_by(['sh6', 'product_description', 'sc_comp'])
    .agg([
        pl.sum('potencial_total_sc').alias('potencial_total_sc'),
        pl.sum('exportacoes_sc').alias('exportacoes_sc'),
        pl.sum('potencial_nao_realizado_sc').alias('potencial_nao_realizado_sc'),
    ])
    .sort('potencial_total_sc', descending=True)
)

df_resumo = pl.DataFrame({
    'metrica': [
        'linhas_sh6_parceiro',
        'linhas_sh6_total',
        'potencial_total_sc',
        'exportacoes_sc',
        'potencial_nao_realizado_sc',
    ],
    'valor': [
        float(df_sc_detail.height),
        float(df_sc_sh6.height),
        float(df_sc_detail['potencial_total_sc'].sum()),
        float(df_sc_detail['exportacoes_sc'].sum()),
        float(df_sc_detail['potencial_nao_realizado_sc'].sum()),
    ],
})

df_sc_detail.write_csv(out_csv_detail)
df_sc_sh6.write_csv(out_csv_sh6)

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    df_resumo.to_pandas().to_excel(writer, sheet_name='resumo', index=False)
    df_sc_sh6.to_pandas().to_excel(writer, sheet_name='total_por_sh6', index=False)
    df_sc_detail.to_pandas().to_excel(writer, sheet_name='sh6_por_parceiro', index=False)

print(f'Arquivo xlsx exportado: {out_xlsx}')
print(f'Arquivo csv detalhe exportado: {out_csv_detail}')
print(f'Arquivo csv total SH6 exportado: {out_csv_sh6}')
print(f'Linhas SH6 x parceiro: {df_sc_detail.height}')
print(f'Linhas SH6 total: {df_sc_sh6.height}')

df_resumo
df_sc_detail.head(20)